<a href="https://colab.research.google.com/github/sita-aghasoy33/basic-llm-tutorials/blob/main/RAG_without_functions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌳 RAG — PDF → Markdown + Recursive (token-əsaslı) Splitting - without functions

In [1]:
!pip install -q pymupdf4llm sentence-transformers faiss-cpu transformers accelerate

import pymupdf4llm
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import re
from transformers import pipeline


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.4/176.4 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 30.9 MB/s eta 0:00:00


In [11]:
!pip install -q -U "bitsandbytes>=0.46.1"
!pip show bitsandbytes | grep Version

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.7 MB/s eta 0:00:00
Version: 0.50.2


In [12]:
from transformers import BitsAndBytesConfig
generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-7B-Instruct",
    device_map="auto",
    model_kwargs={"quantization_config": BitsAndBytesConfig(load_in_4bit=True)},
)
print(generator.model.device)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

cuda:0


In [2]:
PDF_PATH = "/content/sirket_siyaseti_normal.pdf"  # öz PDF faylınızın yolu ilə əvəz edin

md_text = pymupdf4llm.to_markdown(PDF_PATH)

print(f"Ümumi simvol sayı: {len(md_text)}")
print(md_text[:500])


=== Document parser messages ===
Using Tesseract for OCR processing.
Ümumi simvol sayı: 1446
# **Uzaqdan İşləmə Siyasəti — Nordic Tech MMC** 

Sənəd nömrəsi: HR-POL-014  |  Təsdiq tarixi: 2026-01-15  |  Versiya: 2.1 

## **1. Məqsəd** 

Bu sənəd Nordic Tech MMC daxilində uzaqdan işləmə qaydalarını, işçilərin hüquq və vəzifələrini, həmçinin texniki və təhlükəsizlik tələblərini müəyyən edir. Siyasət bütün tam və yarım ştat işçilərinə şamil olunur. 

## **2. Uyğunluq şərtləri** 

Uzaqdan işləmək üçün işçi ən azı 6 ay şirkətdə çalışmış olmalı, birbaşa rəhbərin yazılı təsdiqini almalı və İT 


In [4]:
embedding_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [5]:
md_text = md_text
pattern = r"(?=^#{1,3} .+$)"
sections = re.split(pattern, md_text, flags=re.MULTILINE)

print(f"{len(sections)} bölmə tapıldı")

7 bölmə tapıldı


In [8]:
chunk_embeddings = embedding_model.encode(sections, convert_to_numpy=True, show_progress_bar=True)
print("Embedding ölçüsü:", chunk_embeddings.shape)

dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(chunk_embeddings)
print(f"İndeksdə {index.ntotal} chunk var, hər biri {dimension} ölçülü vektor")

print(len(embedding_model.tokenizer.encode(sections, add_special_tokens=False)))
print(embedding_model.tokenizer.encode(sections, add_special_tokens=False))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding ölçüsü: (7, 384)
İndeksdə 7 chunk var, hər biri 384 ölçülü vektor
7
[[], [468, 16459, 1062, 596, 108923, 20468, 114174, 38190, 7, 53507, 292, 98340, 36630, 119061, 25442, 159, 36964, 71, 161082, 172, 12, 22439, 9, 120148, 55667, 617, 6, 58745, 29202, 7, 65299, 22521, 12, 387, 4046, 45669, 1837, 6, 58745, 1446, 10887, 12, 34513], [173894, 16459, 418, 5, 19026, 864, 12749, 71, 25442, 667, 91087, 98340, 36630, 119061, 197919, 113422, 549, 59114, 6749, 85175, 4242, 4, 33178, 5846, 63385, 530, 131217, 943, 4, 58957, 6, 110384, 530, 150780, 182056, 943, 25831, 7284, 5, 38190, 7, 9485, 7609, 1481, 530, 80107, 6, 1759, 3935, 33178, 20206, 28339, 12397, 25329, 5], [173894, 16459, 5442, 58039, 85110, 27700, 118211, 14, 25442, 10054, 11, 108923, 4659, 114048, 2385, 33178, 9625, 295, 1057, 305, 537, 127668, 1477, 22212, 9027, 117007, 4, 175031, 59774, 19, 89675, 45056, 943, 144, 31123, 530, 6, 104886, 54946, 3266, 41083, 25366, 150780, 99206, 257, 4242, 33369, 46688, 5, 159, 9291, 864, 2

In [14]:
query = "Write your prompt"
k = 3
query_embedding = embedding_model.encode([query], convert_to_numpy=True)
distances, indices = index.search(query_embedding, k)
print([sections[i] for i in indices[0]])

['## **2. Uyğunluq şərtləri** \n\nUzaqdan işləmək üçün işçi ən azı 6 ay şirkətdə çalışmış olmalı, birbaşa rəhbərin yazılı təsdiqini almalı və İT departamentinin tələb etdiyi təhlükəsizlik təlimatlarını keçməlidir. Sınaq müddətində olan işçilər bu imkandan istifadə edə bilməzlər. \n\n', '', '# **Uzaqdan İşləmə Siyasəti — Nordic Tech MMC** \n\nSənəd nömrəsi: HR-POL-014  |  Təsdiq tarixi: 2026-01-15  |  Versiya: 2.1 \n\n']


In [15]:
SYSTEM_INSTRUCTIONS = """

Sən şirkət sənədləri üzrə köməkçisən. İstifadəçinin sualına yalnız verilən KONTEKST əsasında, Azərbaycan dilində cavab ver.

Qaydalar:
1. Yalnız KONTEKST-dəki məlumatdan istifadə et, öz biliyini əlavə etmə.
2. KONTEKST-də suala aid məlumat varsa (hətta qismən), mütləq onunla cavab ver.
3. Yalnız KONTEKST-də suala aid heç bir məlumat olmadıqda yaz: "Bu sualın cavabı verilən sənəddə yoxdur."
4. Cavab qısa və aydın olsun (lazım gələrsə bullet ilə). Uydurma fakt, rəqəm və ya tarix yazma.

"""


context_chunks = [sections[i] for i in indices[0]]
context = "\n\n---\n\n".join(context_chunks)

messages = [
        {"role": "system", "content": SYSTEM_INSTRUCTIONS},
        {"role": "user", "content": f"KONTEKST:\n{context}\n\nSUAL: {query}"},
    ]


result = generator(
        messages,
        max_new_tokens=200,
        do_sample=False,          # deterministik cavab
        repetition_penalty=1.1,   # eyni cümləni təkrarlamasın
    )

# chat formatında generated_text mesajlar siyahısıdır -> sonuncu = modelin cavabı
answer = result[0]["generated_text"][-1]["content"].strip()
print({"answer": answer, "sources": context_chunks, "messages": messages})

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'answer': 'Bu sualın cavabı verilən sənəddə yoxdur.', 'sources': ['## **2. Uyğunluq şərtləri** \n\nUzaqdan işləmək üçün işçi ən azı 6 ay şirkətdə çalışmış olmalı, birbaşa rəhbərin yazılı təsdiqini almalı və İT departamentinin tələb etdiyi təhlükəsizlik təlimatlarını keçməlidir. Sınaq müddətində olan işçilər bu imkandan istifadə edə bilməzlər. \n\n', '', '# **Uzaqdan İşləmə Siyasəti — Nordic Tech MMC** \n\nSənəd nömrəsi: HR-POL-014  |  Təsdiq tarixi: 2026-01-15  |  Versiya: 2.1 \n\n'], 'messages': [{'role': 'system', 'content': '\n\nSən şirkət sənədləri üzrə köməkçisən. İstifadəçinin sualına yalnız verilən KONTEKST əsasında, Azərbaycan dilində cavab ver.\n\nQaydalar:\n1. Yalnız KONTEKST-dəki məlumatdan istifadə et, öz biliyini əlavə etmə.\n2. KONTEKST-də suala aid məlumat varsa (hətta qismən), mütləq onunla cavab ver.\n3. Yalnız KONTEKST-də suala aid heç bir məlumat olmadıqda yaz: "Bu sualın cavabı verilən sənəddə yoxdur."\n4. Cavab qısa və aydın olsun (lazım gələrsə bullet ilə). Uydur